<a href="https://colab.research.google.com/github/axelit19/Algoritmos-de-IA/blob/main/40_UNAM_AI_Actividad_Laboratorio_5_11_algoritmos_clase.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<center>

$\Huge \textbf{Universidad Nacional Autónoma de México}$  
$\Huge \textbf{Facultad de Ciencias}$  
<p align="center">
  <img src="https://www.icat.unam.mx/wp-content/uploads/2021/11/Copia-de-LogoUNAM.-Azul.-Fondo-transparente.png" alt="UNAM" width="200"/>
</p>

<hr style="height:3px; background-color:#0B6E4F; border:none;"/>


$\LARGE \textbf{Inteligencia Artificial}$  

$\Large \textit{Laboratorio 2.3}$  


\begin{array}{rl}
\textbf{Docente:} & Dra. Jessica Sarahi Méndez Rincón \\[6pt]
\textbf{Ayudante de laboratorio:} & Diego Eduardo Peña Villegas \\[6pt]
\textbf{Alumna:} & Marisol Luna Méndez \\[6pt]
\textbf{Fecha de realización:} & 25/02/2026
\end{array}

</center>

In [ ]:
!pip install Word2Vec

In [ ]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 68.3 MB/s eta 0:00:00


# Algoritmo  de Distancia de Levenshtein es un  

In [ ]:
def levenshtein_futbol(palabra_a, palabra_b):
    """
    Calcula la distancia de edición entre dos términos.
    Ejemplo: 'Messi' vs 'Messi' (0), 'Gol' vs 'Gool' (1)
    """
    n = len(palabra_a)
    m = len(palabra_b)

    # 1. Crear la matriz inicializada en ceros
    matriz = [[0 for _ in range(m + 1)] for _ in range(n + 1)]

    # 2. Casos base: transformar cadena a vacío o viceversa
    for i in range(n + 1):
        matriz[i][0] = i
    for j in range(m + 1):
        matriz[0][j] = j

    # 3. Llenado de la matriz con programación dinámica
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            # Si los caracteres son iguales, el costo es 0
            if palabra_a[i-1] == palabra_b[j-1]:
                costo = 0
            else:
                costo = 1

            # Buscamos el mínimo entre las 3 operaciones posibles
            matriz[i][j] = min(
                matriz[i-1][j] + 1,      # Eliminación
                matriz[i][j-1] + 1,      # Inserción
                matriz[i-1][j-1] + costo # Sustitución
            )

    return matriz[n][m]

# --- PRUEBA PARA CLASE ---
termino_busqueda = "Gool"
termino_real = "Gol"

distancia = levenshtein_futbol(termino_busqueda, termino_real)
print(f"La distancia entre '{termino_busqueda}' y '{termino_real}' es: {distancia}")
# Resultado esperado: 1

La distancia entre 'Gool' y 'Gol' es: 1


#Word2Vec

Ejemplo en Código (Python + Gensim)

Usaríamos la librería gensim. Imaginemos que ya entrenamos el modelo con noticias de la Liga MX o de la Champions:

In [ ]:
from gensim.models import Word2Vec

# Un 'corpus' de ejemplo (muy simplificado)
entrenamiento_futbol = [
    ["el", "delantero", "pateó", "al", "arco"],
    ["el", "atacante", "pateó", "al", "marco"],
    ["el", "portero", "detuvo", "el", "balón"],
    ["el", "arquero", "atajó", "la", "pelota"]
]

# Entrenamos el modelo
modelo = Word2Vec(sentences=entrenamiento_futbol, vector_size=100, window=2, min_count=1)

# 1. Encontrar similitud
similitud = modelo.wv.similarity('delantero', 'atacante')
print(f"Similitud entre delantero y atacante: {similitud:.2f}")

# 2. ¿Cuál no encaja en la alineación?
no_encaja = modelo.wv.doesnt_match(["delantero", "atacante", "arquero", "pizza"])
print(f"El infiltrado es: {no_encaja}")

Similitud entre delantero y atacante: -0.16
El infiltrado es: atacante


# Word2Vec Algoritmo desde Cero

In [ ]:
import numpy as np

def softmax(x):
    e_x = np.exp(x - np.max(x))
    return e_x / e_x.sum(axis=0)

class Word2VecSimplified:
    def __init__(self, vocab_size, embed_dist):
        self.n = embed_dist # Dimensiones
        self.v = vocab_size # Tamaño vocabulario
        # Inicialización de pesos (He initialization)
        self.w1 = np.random.randn(self.v, self.n)
        self.w2 = np.random.randn(self.n, self.v)

    def forward(self, x):
        # x es un vector one-hot o el índice de la palabra
        h = self.w1[x]
        u = np.dot(h, self.w2)
        y_c = softmax(u)
        return y_c, h, u

    def backward(self, e, h, x):
        # Gradientes para actualizar los pesos
        dw2 = np.outer(h, e)
        dw1 = np.dot(self.w2, e.T)

        # Actualización simple (tasa de aprendizaje 0.01)
        self.w2 = self.w2 - (0.01 * dw2)
        self.w1[x] = self.w1[x] - (0.01 * dw1.T)

# Ejemplo de uso conceptual:
# 1. Definimos vocabulario: {'gol': 0, 'messi': 1, 'futbol': 2}
# 2. Si entra 'messi' (index 1) y el contexto es 'gol' (index 0)
modelo = Word2VecSimplified(vocab_size=3, embed_dist=2)
y_pred, h, u = modelo.forward(1)

# El error es (predicción - real) donde real es un one-hot del contexto
error = y_pred.copy()
error[0] -= 1

modelo.backward(error, h, 1)
print("Vector actualizado de 'Messi':", modelo.w1[1])

Vector actualizado de 'Messi': [0.67723212 0.00856268]


<hr/>
<footer style="text-align:center; font-size:12px; color:gray;">
© 2026 UNAM Facultado de Ciencias – Todos los derechos reservados

</footer>